# Karlsson dark segments — correlation of latent variables to biological features

Workflow:

1. Use cells from previous Karlsson analysis notebooks to generate data files (`karlsson_dark_segments_128.npz`) and `karlsson_dark_segments_table.pkl`).
2. Normalize each segment by its own maximum, exactly as A7X expects.
3. Load the pretrained A7X-8 model.
4. Encode the **entire** darkness dataset at once. 
5. Build a single 2-part figure:
   - **A.** 5x8 correlation matrix between latent variables and curve-level biological features. 
   - **B.** 5x8 matrix showing plotted correlations (binned, averaged) between value of latent variable and value of curve-level biological feature. 


In [ ]:
from pathlib import Path
import sys
import json
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

import torch

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)

SEED = 42
rng = np.random.default_rng(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

## 0. Generate needed data files, containing dark segment A7X representations.  

With this code, the dark trajectories will be split into non-overlapping 128-point
segments (points 0–127, 128–255, 256–383 of each 385-point trajectory) by the
earlier preprocessing notebook. We simply load those saved outputs here:

- `karlsson_dark_segments_128.npz` — flattened `(N, 128)` segment array plus IDs.
- `karlsson_dark_segments_table.pkl` — segment-level metadata table.

In [ ]:
# ---------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------

# Preferred location of the master pickle
PKL_PATH = Path("analyses/karlsson/outputs/karlsson_master.pkl")

# If the preferred path does not exist, search locally for the file.
if not PKL_PATH.exists():
    print(f"Preferred PKL file not found at: {PKL_PATH.resolve()}")
    print("Searching for karlsson_master.pkl in the current directory...")

    local_hits = []  # Avoid silently loading stale data from another directory.

    if local_hits:
        PKL_PATH = local_hits[0]
        print(f"Found master PKL at: {PKL_PATH.resolve()}")
    else:
        raise FileNotFoundError(
            "\nCould not find karlsson_master.pkl.\n\n"
            "Please run Karlsson_dark_preprocessing.ipynb first.\n"
            "The builder notebook should create:\n"
            "    analyses/karlsson/outputs/karlsson_master.pkl\n\n"
            "Also check that you are running this notebook from the "
            "correct project directory."
        )

# ---------------------------------------------------------------------
# Output directory
# ---------------------------------------------------------------------

OUTPUT_DIR = PKL_PATH.parent / "dark_segment_a7x_pca_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Loading master pkl from: {PKL_PATH.resolve()}")
print(f"Outputs will be saved to: {OUTPUT_DIR.resolve()}")

# ---------------------------------------------------------------------
# Load master pickle
# ---------------------------------------------------------------------

try:
    with open(PKL_PATH, "rb") as f:
        master = pickle.load(f)
except Exception as e:
    raise RuntimeError(
        f"\nThe file was found, but could not be loaded:\n"
        f"    {PKL_PATH.resolve()}\n\n"
        f"Original error: {type(e).__name__}: {e}"
    ) from e

# ---------------------------------------------------------------------
# Validate master structure
# ---------------------------------------------------------------------

if not isinstance(master, dict):
    raise TypeError(
        f"\nExpected the pickle to contain a dictionary, "
        f"but found: {type(master).__name__}"
    )

if "tables" not in master:
    raise KeyError(
        "\nThe loaded master pickle does not contain the expected "
        "'tables' key.\n"
        f"Available top-level keys: {list(master.keys())}"
    )

print("Top-level keys:", list(master.keys()))
print("Tables:", list(master["tables"].keys()))


In [ ]:
traj = master["tables"]["trajectories"].copy()

# Ensure trajectory_id is present as a column.
if "trajectory_id" not in traj.columns:
    if traj.index.name == "trajectory_id":
        traj = traj.reset_index()
    else:
        traj["trajectory_id"] = traj.index.astype(str)
traj["trajectory_id"] = traj["trajectory_id"].astype(str)

# Use the index built by the master notebook when possible.
try:
    ids_dark = master["indexes"]["by_treatment"][("darkness",)]
    dark = traj[traj["trajectory_id"].isin(ids_dark)].copy()
except Exception:
    dark = traj[traj["treatment"].astype(str).eq("darkness")].copy()

# Reproducible order.
sort_cols = [c for c in ["organism", "species_code", "replicate", "plate_type", "well", "trajectory_id"] if c in dark.columns]
if sort_cols:
    dark = dark.sort_values(sort_cols).reset_index(drop=True)
else:
    dark = dark.sort_values("trajectory_id").reset_index(drop=True)

print("Dark trajectories:", dark.shape)
print("Columns:", list(dark.columns))

# Check point counts.
dark["n_points_original"] = dark["values"].apply(lambda x: len(np.asarray(x)))
print(dark["n_points_original"].describe())
print("Unique point counts:", sorted(dark["n_points_original"].unique())[:20])

display(dark.head())

In [ ]:
SEGMENT_LENGTH = 128
N_SEGMENTS_PER_TRAJECTORY = 3
REQUIRED_POINTS = SEGMENT_LENGTH * N_SEGMENTS_PER_TRAJECTORY  # 384

segment_rows = []
segment_values = []
segment_times = []
skipped = []

for _, row in dark.iterrows():
    values = np.asarray(row["values"], dtype=np.float32)
    times = np.asarray(row.get("time_hours", np.arange(len(values))), dtype=np.float32)
    if len(values) < REQUIRED_POINTS:
        skipped.append((row["trajectory_id"], len(values)))
        continue

    # Drop any points beyond the first 384, including point 385 when present.
    values_384 = values[:REQUIRED_POINTS]
    times_384 = times[:REQUIRED_POINTS]

    for seg_idx in range(N_SEGMENTS_PER_TRAJECTORY):
        start = seg_idx * SEGMENT_LENGTH
        end = start + SEGMENT_LENGTH
        seg_values = values_384[start:end]
        seg_times = times_384[start:end]

        meta = row.drop(labels=["values", "time_hours"], errors="ignore").to_dict()
        source_id = str(row["trajectory_id"])
        meta.update({
            "source_trajectory_id": source_id,
            "segment_id": f"{source_id}__seg{seg_idx + 1}",
            "segment_index": seg_idx + 1,
            "segment_start_idx_0based": start,
            "segment_end_idx_exclusive_0based": end,
            "segment_length": SEGMENT_LENGTH,
            "dropped_after_idx_0based": REQUIRED_POINTS,
            "n_points_used_from_original": REQUIRED_POINTS,
            "n_points_original": len(values),
            "segment_time_start": float(seg_times[0]) if len(seg_times) else np.nan,
            "segment_time_end": float(seg_times[-1]) if len(seg_times) else np.nan,
            "segment_min": float(np.nanmin(seg_values)),
            "segment_max": float(np.nanmax(seg_values)),
            "segment_mean": float(np.nanmean(seg_values)),
            "segment_auc_index_space": float(np.trapz(seg_values)),
        })
        segment_rows.append(meta)
        segment_values.append(seg_values)
        segment_times.append(seg_times)

segments_table = pd.DataFrame(segment_rows)
X_segments = np.vstack(segment_values).astype(np.float32)
T_segments = np.vstack(segment_times).astype(np.float32)

print("Segments table:", segments_table.shape)
print("Flattened segment array:", X_segments.shape)
print("Segment time array:", T_segments.shape)
print("Skipped trajectories:", len(skipped))
if skipped:
    print("First skipped examples:", skipped[:10])

assert X_segments.shape[0] == segments_table.shape[0]
assert X_segments.shape[1] == SEGMENT_LENGTH
assert not np.isnan(X_segments).any(), "NaN values detected in segment matrix."

display(segments_table.head())

In [ ]:
# Save the segment labels / metadata table.
segments_table_path_pkl = OUTPUT_DIR / "karlsson_dark_segments_table.pkl"
segments_table_path_csv = OUTPUT_DIR / "karlsson_dark_segments_table.csv.gz"
segments_table.to_pickle(segments_table_path_pkl)
segments_table.to_csv(segments_table_path_csv, index=False)

# Save the arrays.
npz_path = OUTPUT_DIR / "karlsson_dark_segments_128.npz"
np.savez_compressed(
    npz_path,
    X_segments=X_segments,
    T_segments=T_segments,
    segment_id=segments_table["segment_id"].astype(str).to_numpy(),
    source_trajectory_id=segments_table["source_trajectory_id"].astype(str).to_numpy(),
    segment_index=segments_table["segment_index"].to_numpy(),
)

print("Saved:")
print(" -", segments_table_path_pkl)
print(" -", segments_table_path_csv)
print(" -", npz_path)

## 1. Load the preprocessed dark segments.

In [ ]:
CANDIDATE_OUTPUT_DIRS = [Path("analyses/karlsson/outputs/dark_segment_a7x_pca_outputs")]

OUTPUT_DIR = next((p for p in CANDIDATE_OUTPUT_DIRS if (p / "karlsson_dark_segments_128.npz").exists()), None)

if OUTPUT_DIR is None:
    local_hits = []  # Avoid silently loading stale segment files.
    if local_hits:
        OUTPUT_DIR = local_hits[0].parent

if OUTPUT_DIR is None:
    raise FileNotFoundError(
        "Could not find karlsson_dark_segments_128.npz. "
        "Run the preprocessing/segmentation notebook first to generate the "
        "128-point dark segments used here."
    )

print("Using preprocessed segments from:", OUTPUT_DIR.resolve())

seg_npz = np.load(OUTPUT_DIR / "karlsson_dark_segments_128.npz", allow_pickle=True)
X_segments = seg_npz["X_segments"].astype(np.float32)  # (N, 128) raw curves
T_segments = seg_npz["T_segments"].astype(np.float32)  # (N, 128) real time (hours)
segment_id = seg_npz["segment_id"]
source_trajectory_id = seg_npz["source_trajectory_id"]
segment_index = seg_npz["segment_index"]

segments_table = pd.read_pickle(OUTPUT_DIR / "karlsson_dark_segments_table.pkl")

print("X_segments:", X_segments.shape)
print("segments_table:", segments_table.shape)
assert X_segments.shape[0] == segments_table.shape[0]
assert X_segments.shape[1] == 128

## 2. Normalize segments for A7X

Each segment is normalized by its own maximum before A7X encoding, following the
existing convention. The raw per-segment maximum is kept so reconstructions can be
mapped back to original curve scale.

In [ ]:
def prepare_curves_for_a7x(curves):
    """Normalize curves by per-curve maximum and return tensor plus raw maxima."""
    curves = np.asarray(curves, dtype=np.float32)
    if curves.ndim == 1:
        curves = curves[None, :]
    if curves.shape[1] != 128:
        raise ValueError(f"Expected curves with shape (N, 128), got {curves.shape}")

    raw_max = np.nanmax(curves, axis=1, keepdims=True).astype(np.float32)
    denom = np.where(raw_max > 0, raw_max, 1.0).astype(np.float32)
    curves_norm = curves / denom
    x = torch.from_numpy(curves_norm).unsqueeze(1)  # (N, 1, 128)
    return x, raw_max, curves_norm.astype(np.float32)

X_tensor, raw_max, X_segments_norm = prepare_curves_for_a7x(X_segments)
print("X_segments_norm:", X_segments_norm.shape)
print("raw_max:", raw_max.shape)

## 3. Load pretrained A7X-8

Requirements on the machine running this notebook:

- The repository source code should be available at `./src`, so that
  `ml.models.autoencoder7x` can be imported.
- Internet or cached Hugging Face files are needed for `you-lab/autoencoder7x-08`
  unless the model is already cached.

In [ ]:
from huggingface_hub import hf_hub_download

repo_dir = Path("./")
sys.path.append(str(repo_dir / "src"))

try:
    from ml.models.autoencoder7x import Autoencoder7XModel
except ModuleNotFoundError as e:
    raise ModuleNotFoundError(
        "Could not import Autoencoder7XModel from ml.models.autoencoder7x. "
        "Run this notebook from the project root where ./src is available, or update repo_dir above."
    ) from e

REPO_ID = "you-lab/autoencoder7x-08"


def load_a7x_model(repo_id=REPO_ID, device=DEVICE):
    config_path = hf_hub_download(repo_id=repo_id, filename="config.json")
    weights_path = hf_hub_download(repo_id=repo_id, filename="pytorch_model.bin")

    with open(config_path, "r") as f:
        config = json.load(f)
    config.pop("model_type", None)

    model = Autoencoder7XModel(config)
    state_dict = torch.load(weights_path, map_location=device)
    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()
    return model

fm = load_a7x_model()
fm.eval()
print(fm.__class__.__name__, "loaded")

## 4. Compress the entire dataset with A7X

Every dark segment is encoded to
obtain its 8-dimensional latent `mu`, `log_var`, and normalized reconstruction.

In [ ]:
def a7x_forward_with_mu_logvar(model, xb):
    """Return reconstruction, mu, and log_var from an A7X forward pass."""
    out = model(xb)

    if isinstance(out, dict):
        recon = out.get("recon", out.get("reconstruction", out.get("x_recon", None)))
        mu = out.get("mu", out.get("z", out.get("latent", None)))
        log_var = out.get("log_var", out.get("logvar", None))
        if recon is not None and mu is not None and log_var is not None:
            return recon, mu, log_var

    if isinstance(out, (tuple, list)):
        if len(out) >= 3:
            recon, mu, log_var = out[0], out[1], out[2]
            return recon, mu, log_var
        if len(out) == 2:
            raise RuntimeError(
                "A7X returned only two outputs. Need log_var for this analysis."
            )

    raise RuntimeError(
        "Unexpected A7X forward output. Inspect model.forward output and update "
        "a7x_forward_with_mu_logvar."
    )


def encode_segments_a7x(curves, model, batch_size=1024, device=DEVICE):
    x, raw_max, curves_norm = prepare_curves_for_a7x(curves)
    mu_list, logvar_list, recon_list = [], [], []

    model.eval()
    with torch.no_grad():
        for start in range(0, x.shape[0], batch_size):
            xb = x[start:start + batch_size].to(device)
            recon, mu, log_var = a7x_forward_with_mu_logvar(model, xb)
            recon_list.append(recon.detach().cpu().view(recon.shape[0], -1).numpy())
            mu_list.append(mu.detach().cpu().view(mu.shape[0], -1).numpy())
            logvar_list.append(log_var.detach().cpu().view(log_var.shape[0], -1).numpy())

    recon_norm = np.vstack(recon_list).astype(np.float32)
    mu = np.vstack(mu_list).astype(np.float32)
    log_var = np.vstack(logvar_list).astype(np.float32)
    return mu, log_var, recon_norm, raw_max, curves_norm


mu_a7x, logvar_a7x, recon_a7x_norm, raw_max, X_segments_norm = encode_segments_a7x(
    X_segments, fm, batch_size=1024, device=DEVICE,
)

# Reconstructions mapped back to original curve scale.
recon_a7x_raw = recon_a7x_norm * raw_max

print("mu_a7x:", mu_a7x.shape)
print("logvar_a7x:", logvar_a7x.shape)
print("recon_a7x_norm:", recon_a7x_norm.shape)
assert mu_a7x.shape[1] == 8, "Expected A7X-8 to produce 8 latent dimensions."

In [ ]:
latent_npz_path = OUTPUT_DIR / "karlsson_dark_segments_a7x_latents_reconstruction.npz"
np.savez_compressed(
    latent_npz_path,
    mu=mu_a7x,
    log_var=logvar_a7x,
    recon_norm=recon_a7x_norm,
    raw_max=raw_max,
    X_segments_norm=X_segments_norm,
    segment_id=segment_id,
    source_trajectory_id=source_trajectory_id,
    segment_index=segment_index,
)
print("Saved:", latent_npz_path)

## 5. Curve-level summary features vs. A7X latents

For each raw dark-segment curve we first apply a **moving-average filter**
(a centered sliding-window box filter, window size 5) to smooth out
point-to-point measurement noise *within* each curve. From each smoothed
curve we then extract five curve-level summary statistics:

- **Maximum** — peak value of the smoothed curve.
- **AUC** — area under the smoothed curve (trapezoidal integration over real time).
- **Time to maximum** — time (hours) at which the peak occurs.
- **Maximum slope** — largest value of the first derivative (steepest rise).
- **Time to maximum slope** — time (hours) at which the steepest rise occurs.

**Panel A** shows the Spearman correlation between each of these five
features and each of the 8 A7X latent dimensions (`mu`), visualized as a
5×8 correlation heatmap.

**Panel B** visualizes these same feature-vs-latent relationships directly,
as trend lines rather than a single correlation coefficient. This
uses a second smoothing step: for a given latent dimension, we
standardize and sort all segments by their latent value, then compute a
centered rolling mean of the target feature over a window of neighboring
segments in that sorted order. The window size (`TREND_WINDOW_FRAC`, expressed as a fraction of the
segment count) is chosen empirically by checking a couple of representative
(latent, feature) pairs and picking the smallest window at which the trend
shape stabilizes (large enough to
average out sampling noise).

In [ ]:
from scipy.stats import spearmanr

# ============================================================================
# Step 0: Smooth raw curves (TIME-DOMAIN smoothing, applied once per curve)
# ============================================================================
def moving_average_filter(curves, window=5):
    """Smooth each row of `curves` with a centered box filter of size `window`.

    This averages neighboring *time samples*
    within a single curve to remove point-to-point measurement noise before
    any summary statistics (max, AUC, slope, etc.) are computed from it.

    Edges are handled by reflecting the signal so the smoothed curve has the
    same length as the input and doesn't get pulled down near the boundaries.
    """
    if window <= 1:
        return curves.astype(np.float32)
    kernel = np.ones(window, dtype=np.float32) / window
    pad = window // 2
    smoothed = np.empty_like(curves, dtype=np.float32)
    for i in range(curves.shape[0]):
        padded = np.pad(curves[i], pad_width=pad, mode="reflect")
        smoothed[i] = np.convolve(padded, kernel, mode="same")[pad:pad + curves.shape[1]]
    return smoothed


MA_WINDOW = 5
X_segments_smooth = moving_average_filter(X_segments, window=MA_WINDOW)
n_latents = mu_a7x.shape[1]


# ============================================================================
# Step 1: Extract curve-level summary features from each smoothed curve
# ============================================================================
def extract_curve_features(y, t):
    """Extract max, AUC, time-to-max, max slope, and time-to-max-slope for one curve."""
    max_val = np.max(y)
    auc = np.trapz(y, x=t)
    t_max = t[np.argmax(y)]

    slope = np.gradient(y, t)
    max_slope = np.max(slope)
    t_max_slope = t[np.argmax(slope)]

    return max_val, auc, t_max, max_slope, t_max_slope


feature_names = ["max", "auc", "time_to_max", "max_slope", "time_to_max_slope"]
features = np.array([
    extract_curve_features(X_segments_smooth[i], T_segments[i])
    for i in range(X_segments_smooth.shape[0])
])
features_df = pd.DataFrame(features, columns=feature_names)
features_df.insert(0, "segment_id", segment_id)

print("features_df:", features_df.shape)
display(features_df.head())

# ============================================================================
# Step 2: Panel A — feature x latent Spearman correlation matrix
# ============================================================================
n_features = len(feature_names)
feat_latent_corr = np.zeros((n_features, n_latents))
feat_latent_pval = np.zeros((n_features, n_latents))

for fi, fname in enumerate(feature_names):
    for li in range(n_latents):
        rho, pval = spearmanr(features_df[fname].values, mu_a7x[:, li])
        feat_latent_corr[fi, li] = rho
        feat_latent_pval[fi, li] = pval

feature_label_map = {
    "max": "Max value",
    "auc": "AUC",
    "time_to_max": "Time to max",
    "max_slope": "Max slope",
    "time_to_max_slope": "Time to max slope",
}
feature_pretty = [feature_label_map[f] for f in feature_names]
mu_labels = [rf"$\mu_{{{j + 1}}}$" for j in range(n_latents)]

# ============================================================================
# Step 3: Panel B — target vs. latent natural trends (setup)
# ============================================================================
target_cols = ["auc", "max", "time_to_max", "max_slope", "time_to_max_slope"]
target_labels = [feature_label_map[t] for t in target_cols]  # single-line, matches Panel A style
n_targets = len(target_cols)


# ============================================================================
# Step 4: Population-level trend smoothing (DIFFERENT from Step 0!)
# ============================================================================
def latent_target_trend(latent_vals, target_vals, window_frac):
    """Sort segments by (standardized) latent value, then smooth the target
    feature with a centered rolling mean *across sorted segments*. After sorting all
    segments by their latent value, this bins/averages the target feature
    over a rolling window of neighboring segments (in latent-value order,
    not time order). This turns a noisy per-segment scatter (one point per
    curve's max/AUC/etc. vs. its latent value) into a visualizable trend
    line.
    """
    z = (latent_vals - latent_vals.mean()) / latent_vals.std()
    order = np.argsort(z)
    z_sorted = z[order]
    y_sorted = target_vals[order]

    window = max(int(window_frac * len(z_sorted)), 5)
    y_smooth = (
        pd.Series(y_sorted)
        .rolling(window=window, center=True, min_periods=1)
        .mean()
        .to_numpy()
    )
    return z_sorted, y_smooth


# ============================================================================
# Step 5: Pick TREND_WINDOW_FRAC empirically (diagnostic plot)
# ============================================================================
window_fracs_to_test = [0.01, 0.02, 0.05, 0.1, 0.2]
colors = plt.cm.viridis(np.linspace(0, 0.85, len(window_fracs_to_test)))

# Pick a couple of representative (latent, target) pairs to inspect --
# ideally whichever cells showed the strongest |correlation| in Panel A.
pairs_to_check = [
    (0, "max_slope"),          # mu_1 vs. max slope
    (2, "time_to_max_slope"),  # mu_3 vs. time to max slope
]

fig_diag, axes_diag = plt.subplots(1, len(pairs_to_check), figsize=(7 * len(pairs_to_check), 5))
if len(pairs_to_check) == 1:
    axes_diag = [axes_diag]

for ax, (latent_idx, target_col) in zip(axes_diag, pairs_to_check):
    latent_vals = mu_a7x[:, latent_idx]
    target_vals = features_df[target_col].to_numpy()
    z = (latent_vals - latent_vals.mean()) / latent_vals.std()

    ax.scatter(z, target_vals, s=6, alpha=0.15, color="gray", label="Raw data")

    # Overlay trend lines at several candidate window fractions so we can see
    # where the shape stops changing (i.e., stabilizes) as window_frac grows.
    for frac, color in zip(window_fracs_to_test, colors):
        z_sorted, y_smooth = latent_target_trend(latent_vals, target_vals, window_frac=frac)
        ax.plot(z_sorted, y_smooth, color=color, lw=2.2, label=f"frac={frac}")

    ax.set_xlabel(rf"$\mu_{{{latent_idx + 1}}}$ (std units)", fontsize=12)
    ax.set_ylabel(feature_label_map[target_col], fontsize=12)
    ax.legend(fontsize=8, loc="best")
    ax.set_title(f"$\\mu_{{{latent_idx + 1}}}$ vs. {feature_label_map[target_col]}", fontsize=13)

plt.tight_layout()
plt.show()

# Chosen fraction: 
TREND_WINDOW_FRAC = 0.05

# ============================================================================
# Step 6: Build the final two-panel figure
# ============================================================================
FS_TITLE = 16
FS_LABEL = 14
FS_TICK = 12
FS_ANNOT = 11
FS_ROWLABEL = 15
FS_PANEL_TAG = 22

fig = plt.figure(figsize=(3.2 * n_latents, 9.5 + 3.6 * n_targets))  # wider + taller per panel
outer = gridspec.GridSpec(2, 1, height_ratios=[2.2, 1.4 * n_targets], hspace=0.28)

# --- Panel A: feature x latent correlation heatmap ---
gsA = gridspec.GridSpecFromSubplotSpec(1, 1, subplot_spec=outer[0])
axA = fig.add_subplot(gsA[0, 0])
im = axA.imshow(feat_latent_corr, vmin=-1, vmax=1, cmap="RdBu_r", aspect=0.6)  # taller cells
axA.set_xticks(range(n_latents))
axA.set_xticklabels(mu_labels, fontsize=FS_TICK)
axA.set_yticks(range(n_features))
axA.set_yticklabels(feature_pretty, fontsize=FS_TICK)
axA.set_anchor('W')
fig.canvas.draw()  # force aspect-based shrinking to resolve before placing colorbar

posA_final = axA.get_position()
cax = fig.add_axes([posA_final.x1 + 0.05, posA_final.y0, 0.015, posA_final.height])
cbar = fig.colorbar(im, cax=cax)
cbar.set_label("Spearman correlation", fontsize=FS_LABEL)
cbar.ax.tick_params(labelsize=FS_TICK)

for fi in range(n_features):
    for li in range(n_latents):
        axA.text(li, fi, f"{feat_latent_corr[fi, li]:.2f}", ha="center", va="center",
                  fontsize=FS_ANNOT, color="black")

# --- Panel B: target vs. latent trend grid ---
gsB = gridspec.GridSpecFromSubplotSpec(n_targets, n_latents, subplot_spec=outer[1],
                                        wspace=0.35, hspace=0.45)
axesB = np.empty((n_targets, n_latents), dtype=object)
for ti, (tcol, tlabel) in enumerate(zip(target_cols, target_labels)):
    target_vals = features_df[tcol].to_numpy()
    for j in range(n_latents):
        ax = fig.add_subplot(gsB[ti, j])
        axesB[ti, j] = ax
        z_sorted, y_smooth = latent_target_trend(mu_a7x[:, j], target_vals,
                                                   window_frac=TREND_WINDOW_FRAC)
        ax.plot(z_sorted, y_smooth, color="#4C72B0", lw=2.2)
        ax.tick_params(labelsize=FS_TICK)

        if ti == n_targets - 1:  # bottom row only: identify the latent
            ax.set_xlabel(mu_labels[j], fontsize=FS_LABEL)

for ti, tlabel in enumerate(target_labels):
    pos = axesB[ti, 0].get_position(fig)
    fig.text(0.045, (pos.y0 + pos.y1) / 2, tlabel, fontsize=FS_ROWLABEL, fontweight="normal",
              ha="center", va="center", rotation=0)

# --- Panel labels A / B ---
posA = axA.get_position(fig)
posB0 = axesB[0, 0].get_position(fig)
fig.text(0.02, posA.y1 + 0.01, "A", fontsize=FS_PANEL_TAG, fontweight="bold")
fig.text(0.02, posB0.y1 + 0.01, "B", fontsize=FS_PANEL_TAG, fontweight="bold")

fig.savefig(OUTPUT_DIR / "a7x_figr5.pdf", dpi=200, bbox_inches="tight")
plt.show()
print("Saved:", OUTPUT_DIR / "a7x_figr5.pdf")